# Retrieval Augmented Generation (RAG) Systems with OpenAI

## Setup

This section handles the initial setup, including installing necessary libraries and configuring the OpenAI API client.

In [ ]:
# Install the OpenAI Python library silently
!pip install  -q openai


### OpenAI API Setup

To interact with OpenAI models, we first need to install the `openai` Python library and initialize the client with an API key. Replace `YOUR_API_KEY` with your actual OpenAI API key.

### 1. OpenAI API Setup and Basic Conversational Agent
This section sets up the OpenAI API client and implements a simple conversational agent with memory.

In [ ]:

from openai import OpenAI # Import the OpenAI class from the openai library

# Initialize the OpenAI client with your API key.

client = OpenAI(api_key="Use Your api key for testing")


### Conversational Agent Functions

These functions define the core logic for our conversational agent, including how to process user queries and generate responses using the OpenAI LLM.

In [ ]:
def research_agent(query):
    '''
    A simplified research agent that directly uses the query as the topic
    and splits the query into words to use as keywords.
    '''
    topic = query # The entire query is considered the topic
    keywords = query.split() # The query is split into words to serve as keywords

    return topic, keywords # Returns both the topic and the list of keywords


#### 1. Research Agent (`research_agent`)

This function takes a user query and extracts a topic and keywords. In this simplified version, the entire query serves as the topic, and individual words from the query become the keywords.

#### 2. LLM Response Generator (`llm_response`)

This function constructs a prompt for the Large Language Model (LLM) using the extracted topic, keywords, and a summary of previous interactions from `conversation_memory`. It then calls the OpenAI API to get a generated response.

In [ ]:
def llm_response(topic, keywords):
    '''
    Generates a response from the OpenAI LLM (gpt-4o-mini) based on the current topic
    and previous conversation history.
    '''
    # Extract previous queries from the conversation memory to provide context to the LLM
    previous_queries = [item["query"] for item in conversation_memory]

    # Construct a detailed prompt for the LLM, including previous conversation context
    prompt = f"""
               You are a helpful AI teacher providing insightful information.
               We are continuing an ongoing discussion. Here's a brief summary of our previous inquiries:
               {previous_queries}

               Now, considering our prior conversation, please provide a comprehensive and thoughtful answer to the current query about: "{topic}".
               Please use these keywords to guide your detailed response: {', '.join(keywords)}.
               Aim to make your explanation clear and engaging.
               """

    # Make a call to the OpenAI API to generate a chat completion
    response = client.chat.completions.create(
        model="gpt-4o-mini", # Specify the LLM model to use
        messages=[
            {"role": "system", "content": "You are a helpful AI teacher."},
            {"role": "user", "content": prompt}
        ]
    )

    # Return the content of the LLM's response
    return response.choices[0].message.content

### Main Conversational Loop

This loop continuously prompts the user for input, processes the query, generates an LLM response, and stores the interaction in `conversation_memory`.

In [ ]:

conversation_memory = [] # Initialize an empty list to store conversation history

# Start a continuous conversation loop
while True:

    user_query = input("Enter your topic (type 'exit' to stop): ") # Get user input
    if not user_query.strip():
        print("Please enter a valid query")
        continue


    if user_query.lower() == "exit":
        print("Session ended")
        break # Exit the loop if the user types 'exit'

    # Process the user query using the research agent to get topic and keywords
    topic, keywords = research_agent(user_query)

    # Generate an LLM response based on the topic and conversation history
    try:
       response = llm_response(topic, keywords)
    except Exception as e:
          print("Error:", e)
          continue

    # Append the current query and the LLM's response to conversation memory
    conversation_memory.append({
    "query": user_query,
    "response": response
})

# After the loop ends, print the entire conversation memory
print(conversation_memory)

# Print the last generated AI response
print("\nAI Response:\n")
print(response)


### Displaying Conversation History

After the conversation ends, these cells display the queries from the `conversation_memory` and the final topic and keywords used.

In [ ]:
# Iterate through each item stored in the 'conversation_memory' list.
# Each item is expected to be a dictionary containing at least a 'query' key.
for item in conversation_memory:
    # Print the 'query' value for each item, which represents a user's previous input.
    print(item["query"])


#### Displaying Last Topic and Keywords

In [ ]:
# Print the 'topic' variable. This variable is typically set within the conversational loop.
print('Topic:', topic)
# Print the 'keywords' variable. This variable is typically set within the conversational loop.
print('Keywords:', keywords)


### 1.1 Conversational Agent with Memory and File Storage

This is an enhanced version of the basic conversational agent that includes a `memory_agent` to manage conversation history, limit it to the last 5 interactions, and persist the memory to a `memory.json` file. It also loads previous memory if available.

In [ ]:
import json
from openai import OpenAI

client = OpenAI(api_key="YOUR_API_KEY")

# Load memory from file
try:
    with open("memory.json", "r") as f:
        conversation_memory = json.load(f)
except:
    conversation_memory = []

def research_agent(query):
    topic = query
    keywords = query.split()
    return topic, keywords

def llm_agent(topic, keywords, memory):
    previous_conversation = [
        f"User: {item['query']} | AI: {item['response']}"
        for item in memory
    ]

    prompt = f"""
    Previous conversation:
    {previous_conversation}

    Now answer the current query: {topic}
    Use these keywords: {keywords}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful AI assistant."},
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content

def memory_agent(memory, query, response):
    memory.append(
        {
            "query": query,
            "response": response
        }
    )

    # Memory limit (last 5)
    memory = memory[-5:]

    # Save memory to file
    with open("memory.json", "w") as f:
        json.dump(memory, f)

    return memory

while True:
    user_query = input("Enter query (type 'exit' to stop): ")

    if not user_query.strip():
        print("Please enter a valid query")
        continue

    if user_query.lower() == "exit":
        print("Session ended")
        break

    topic, keywords = research_agent(user_query)

    try:
        response = llm_agent(topic, keywords, conversation_memory)

    except Exception as e:
        print("Error:", e)
        continue

    conversation_memory = memory_agent(
        conversation_memory, user_query, response
    )

    print("\nAI Response:\n")
    print(response)

    print("\nTotal queries:", len(conversation_memory))

### 2.1 Preparing Data for Basic RAG (Text File)

Before demonstrating the RAG system, we need to create a simple text file (`data.txt`) that will serve as our knowledge base. This cell writes some example data to `data.txt`.

### 2. Basic RAG System (Text File)
This section demonstrates a basic Retrieval Augmented Generation (RAG) system using a single text file (`data.txt`) as the knowledge base.

In [ ]:
'''Simple RAG System'''
with open ('data.txt','r') as f:
    retrieved_data = f.read()

# Define a placeholder topic for demonstration purposes
topic = "AI applications" # This will be replaced by user query in a full RAG system implementation

prompt = f"""
You are an AI assistant.
Answer only from the given data.

Data:
{retrieved_data}

Question:
{topic}
"""

# For demonstration, printing the prompt. In a full RAG system, this prompt would be used in an LLM call.
print(prompt)

#### Keyword-Based Data Retrieval

This code block simulates retrieving relevant data from `data.txt` based on keywords present in the user's query. It then constructs the final prompt that will be sent to the LLM.

In [ ]:
def research_agent(query):
    topic = query
    keywords = query.split()
    return topic, keywords

user_query = input("Enter your query: ")

main_keywords = ["healthcare", "education", "automation"]

with open("data.txt", "r") as f:
    data = f.readlines()

relevant_data = []

for line in data:
    for word in main_keywords:
        if word in user_query.lower() and word in line.lower():
            relevant_data.append(line)

retrieved_data = " ".join(relevant_data)

print("Retrieved Data:", retrieved_data)



# Reconstruct the prompt with the new user-defined topic
prompt = f"""
You are an AI assistant.
Answer the question using only the given data.
Do not repeat all the data.
Give a clear and short answer.

Data: {retrieved_data}. Question:
{topic}
"""



### 3.1 Setup for RAG System (PDF File)

To work with PDF documents, we first need to install the `PyPDF2` library for reading PDFs and `reportlab` for creating dummy PDFs. This section also demonstrates creating a sample PDF file and extracting its text.

In [ ]:
retrieved_data = """AI is used in healthcare for diagnosis.
AI is used in education for personalized learning.
AI is used in automation for reducing manual work."""

with open('data.txt', 'w') as f:
    f.write(retrieved_data)

print("Content successfully written to data.txt")

### 2.2 Simple RAG System Implementation (Text File)

This section demonstrates a basic Retrieval Augmented Generation (RAG) system using `data.txt`. It simulates retrieving data based on keywords from a user query and then constructing a prompt for the LLM to answer *only* from the retrieved data.

### 3. RAG System (PDF File)
This section extends the RAG system to work with PDF documents, showing how to extract text and use it for retrieval. First, we'll create a dummy PDF file and install necessary libraries.

In [ ]:
# for pdf file working install libraray
!pip install PyPDF2
import PyPDF2

#### Creating a Dummy PDF File (`data.pdf`)

This cell uses the `reportlab` library to generate a simple PDF file named `data.pdf` containing some sample text. This PDF will serve as our knowledge base for the RAG system.

In [ ]:

pdf_text = ""

with open("data.pdf", "rb") as file:
    reader = PyPDF2.PdfReader(file)

    for page in reader.pages:
        pdf_text += page.extract_text()

print(pdf_text)

### 3.2 Processing PDF Text with RAG Logic

Now, we apply the RAG logic to the extracted PDF text. Similar to the text file example, this section demonstrates how to retrieve relevant lines from the `pdf_text` based on a user query's keywords.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:

!pip install reportlab
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

file_name = "data.pdf"

c = canvas.Canvas(file_name, pagesize=letter)

c.drawString(100, 750, "AI is used in healthcare for diagnosis.")
c.drawString(100, 730, "AI is used in education for personalized learning.")
c.drawString(100, 710, "AI is used in automation for reducing manual work.")

c.save()

print("PDF created")

#### Extracting Text from PDF

This cell reads the content of the `data.pdf` file using `PyPDF2` and extracts all text into the `pdf_text` variable. This extracted text will then be used for retrieval.

Now, let's process the PDF to extract text and apply the RAG logic.

In [ ]:
user_query = "how is ai use in education"

main_keywords = ["healthcare", "education", "automation"]

relevant_data = []

for line in pdf_text.split("\n"):
    for word in main_keywords:
        if word in user_query.lower() and word in line.lower():
            relevant_data.append(line)

retrieved_data = " ".join(relevant_data)

print("Retrieved Data:", retrieved_data)

### 4.1 Preparing Data for Multi-Document RAG System

For a multi-document RAG system, we'll create several specialized text files. Each file will contain information related to a specific topic (e.g., healthcare, education, automation). These files will act as independent knowledge bases.

### 4. Multi-Document RAG System
This section implements a more advanced RAG system capable of selecting and retrieving information from multiple specialized text files based on the user's query. First, we'll create the data files and set up the OpenAI client.

In [ ]:
'''Multiple Documents RAG system'''
from openai import OpenAI

client = OpenAI(api_key="YOUR_API_KEY")

#### 1. Research Agent (`research_agent`)

This agent identifies the topic and keywords from the user's query.

In [ ]:

# Research Agent
def research_agent(query):
    topic = query
    keywords = query.split()
    return topic, keywords

#### 2. File Selection Agent (`get_file_name`)

This agent determines which knowledge base file is most relevant to the user's query based on keywords.

In [ ]:
# Function to decide which file to read
def get_file_name(user_query):
    user_query = user_query.lower()

    if "healthcare" in user_query:
        return "healthcare.txt"
    elif "education" in user_query:
        return "education.txt"
    elif "automation" in user_query:
        return "automation.txt"
    else:
        return None

#### 3. Data Reading Agent (`read_data`)

This agent reads the content from the selected file.

In [ ]:
# Function to read data from file
def read_data(file_name):
    if file_name is None:
        return "No relevant data found."
    try:
        with open(file_name, "r") as f:
            data = f.read()
        return data
    except FileNotFoundError:
        return f"Error: File '{file_name}' not found."

#### 4. LLM Agent (`llm_agent`)

This agent constructs the prompt for the LLM using the retrieved data and the user's question, ensuring the answer is based *only* on the provided data.

In [ ]:
# LLM Agent
def llm_agent(topic, data):
    prompt = f"""
    You are an AI assistant.
    Answer the question using only the given data.
    Give a short and clear answer.

    Data:
    {data}

    Question:
    {topic}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful AI assistant."},
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content

### 4.3 Main Loop for Multi-Document RAG System

This is the core execution loop for the multi-document RAG system. It orchestrates the flow: taking user input, identifying the relevant file, retrieving data, generating an LLM response, and maintaining a conversation memory.

In [ ]:
# Memory
conversation_memory = []

# Main Loop
while True:
    user_query = input("Enter your query (type 'exit' to stop): ")

    if user_query.lower() == "exit":
        print("Session ended.")
        break

    topic, keywords = research_agent(user_query)

    file_name = get_file_name(user_query)

    data = read_data(file_name)

    if "No relevant data found." in data or "Error: File" in data:
        response = data # Directly use the error message as response
    else:
        try:
            response = llm_agent(topic, data)
        except Exception as e:
            print(f"Error generating LLM response: {e}")
            continue

    conversation_memory.append({
        "query": user_query,
        "response": response
    })

    conversation_memory = conversation_memory[-5:] # Keep only the last 5 conversations

    print("\nAI Response:\n", response)
    print("\nConversation Memory (last 5): ")
    for entry in conversation_memory:
        print(f"  User: {entry['query']}\n  AI: {entry['response'][:50]}...")

In [ ]:
conversation_memory.append({
        "query": user_query,
        "response": response
    })

conversation_memory = conversation_memory[-5:]

print("\nAI Response:\n", response)

In [ ]:
with open('healthcare.txt', 'w') as f:
    f.write('AI is used in healthcare for diagnosis and disease detection.\n')
    f.write('AI helps doctors analyze medical data.')
print('healthcare.txt created.')

In [ ]:
with open('education.txt', 'w') as f:
    f.write('AI is used in education for personalized learning.\n')
    f.write('AI helps students learn at their own pace.')
print('education.txt created.')

In [ ]:
with open('automation.txt', 'w') as f:
    f.write('AI is used in automation to reduce manual work.\n')
    f.write('AI helps in smart manufacturing and robotics.')
print('automation.txt created.')

### 4.2 Agents for Multi-Document RAG

This section defines the specialized agents (`research_agent`, `get_file_name`, `read_data`, `llm_agent`) required for the multi-document RAG system. The `get_file_name` function is crucial for directing queries to the appropriate knowledge base.